# Priority map evolution — how DG3 chooses the next fixation

DG3's output is *not* a single saliency map; it's a **priority map** that depends
on the scanpath history. Each new fixation reshapes the next-fixation distribution.

This notebook samples a scanpath step-by-step and visualises the priority map at
selected fixation counts so you can watch the model's expectation shift.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / 'pyproject.toml').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import deepgaze_pytorch
import pysaliency
from pysaliency.models import sample_from_logdensity

from tez_deepgaze.centerbias import load_centerbias_for_image
from tez_deepgaze.device import pick_device, to_device
from tez_deepgaze.instrument import compute_log_density

## PARAMETERS — edit these

In [ ]:
STIM_IDX           = 91                # which image
FIX_INDICES_TO_SHOW = [1, 3, 5, 7]     # at which fixation counts to snapshot the priority map
SEED               = 0
# Start fixation defaults to the image centre. To use a human subject's start point,
# set USE_HUMAN_START = True and SUBJECT_IDX.
USE_HUMAN_START    = False
SUBJECT_IDX        = 0

In [ ]:
device = pick_device()
stimuli, fixations = pysaliency.get_mit1003(location=str(REPO / 'data' / 'mit1003'))
model = to_device(deepgaze_pytorch.DeepGazeIII(pretrained=True), device).eval()

image = np.asarray(stimuli.stimuli[STIM_IDX])
if image.ndim == 2:
    image = np.stack([image] * 3, axis=-1)
H, W = image.shape[:2]
cb = load_centerbias_for_image(H, W)

if USE_HUMAN_START:
    from tez_deepgaze.human_scanpaths import pick_human_scanpath
    h_xs, h_ys, _ = pick_human_scanpath(fixations, STIM_IDX, subject_idx=SUBJECT_IDX)
    start_xy = (float(h_xs[0]), float(h_ys[0]))
else:
    start_xy = (W / 2.0, H / 2.0)
print(f'start point: ({start_xy[0]:.0f}, {start_xy[1]:.0f})')

In [ ]:
# Sample the scanpath one fixation at a time, recording the priority map
# *before* each new sample. Snapshots are taken at the indices in FIX_INDICES_TO_SHOW.
rst = np.random.RandomState(SEED)
xs = [start_xy[0]]
ys = [start_xy[1]]
snapshots = {}
max_fix = max(FIX_INDICES_TO_SHOW)

for step in range(1, max_fix + 1):
    log_d = compute_log_density(model, image, cb, xs, ys, device)
    if step in FIX_INDICES_TO_SHOW:
        snapshots[step] = (np.exp(log_d), list(xs), list(ys))
    nx, ny = sample_from_logdensity(log_d, rst=rst)
    xs.append(float(nx))
    ys.append(float(ny))

print(f'sampled {len(xs)} fixations; snapshots at {sorted(snapshots)}')

In [ ]:
n_panels = len(FIX_INDICES_TO_SHOW)
fig, axes = plt.subplots(1, n_panels, figsize=(5.4 * n_panels, 4.8))
if n_panels == 1:
    axes = [axes]

for ax, step in zip(axes, sorted(snapshots)):
    prio, sxs, sys_ = snapshots[step]
    ax.imshow(image)
    ax.imshow(prio / (prio.max() + 1e-12), cmap='inferno', alpha=0.55)
    ax.plot(sxs, sys_, '-o', color='cyan', linewidth=1.4,
            markersize=7, markeredgecolor='white')
    ax.plot(sxs[0], sys_[0], 'o', color='#39ff14',
            markersize=11, markeredgecolor='black')
    ax.set_title(f'after fixation #{step}\n'
                 f'(history of {step} fixations)')
    ax.axis('off')

fig.suptitle(f'Stim {STIM_IDX} — priority map evolution. '
             f'Green dot = start; cyan trail = sampled history.', fontsize=12)
fig.tight_layout()
plt.show()

## Try this

- Set `USE_HUMAN_START = True` to start the scanpath at a real human's first fixation
  rather than the image centre. The priority map at fixation #1 will be the same
  (since DG3 conditions only on history), but downstream evolution will differ.
- Take more snapshots: `FIX_INDICES_TO_SHOW = [1, 2, 3, 4, 5, 6, 7, 8]`.
- Try a high-consensus image (77) vs a low-consensus image (522) — on the latter the
  priority map stays diffuse no matter how many fixations accumulate.